# Import e caricamento dati

In [28]:
import pickle
import math
import os
import sys
import pandas as pd
import numpy as np
from datetime import datetime
import ast

sys.path.insert(0, os.path.abspath('..'))
from Sorgente.Preprocessing.preprocesing    import Preprocessore
from Sorgente.Indicizzazione.indice_invertito import IndiceInvertito

In [5]:
indice = IndiceInvertito()
indice.carica('../dati/elaborati/indice_fever.pkl')
print(f'Indice caricato:')
print(f'  Termini unici:  {indice.numero_termini:,}')
print(f'  Documenti:      {indice.numero_documenti:,}')
print(f'  Lunghezza media documenti: {indice.lunghezza_media_documenti():.1f} token')

with open('corpus_fever_arricchito.pkl', 'rb') as f:
    corpus_arricchito = pickle.load(f)
print(f'\nCorpus caricato: {len(corpus_arricchito):,} documenti')

Indice caricato:
  Termini unici:  71,837
  Documenti:      14,363
  Lunghezza media documenti: 121.1 token

Corpus caricato: 14,364 documenti


In [7]:
preprocessore = Preprocessore()

df_campione = pd.read_csv('campione_dev_bilanciato.csv')
print(f'Claim caricate: {len(df_campione):,}')

Claim caricate: 1,500


# IDF(Inverse Document Frequency)

In [8]:
def calcola_idf(termine: str, indice: IndiceInvertito) -> float:
    N  = indice.numero_documenti
    df = indice.documenti_con_termine(termine)
    if df == 0:
        return 0.0
    return math.log((N - df + 0.5) / (df + 0.5) + 1)

Test IDF su alcuni termini con diversa frequenza

In [9]:
termini_test = ['film', 'american', 'nba', 'nikolaj', 'broadcast', 'asteroid']
print(f'  N documenti totali: {indice.numero_documenti:,}')
print()
for termine_grezzo in termini_test:
    stem = preprocessore.elabora(termine_grezzo)
    if not stem:
        continue
    s = stem[0]
    df = indice.documenti_con_termine(s)
    idf = calcola_idf(s, indice)
    print(f'  "{termine_grezzo}" (stem: "{s}")')
    print(f'    df={df:,}  IDF={idf:.4f}')

  N documenti totali: 14,363

  "film" (stem: "film")
    df=5,425  IDF=0.9736
  "american" (stem: "american")
    df=6,204  IDF=0.8395
  "nba" (stem: "nba")
    df=138  IDF=4.6416
  "nikolaj" (stem: "nikolaj")
    df=14  IDF=6.8983
  "broadcast" (stem: "broadcast")
    df=529  IDF=3.3005
  "asteroid" (stem: "asteroid")
    df=22  IDF=6.4590


Verifica della relazione inversa: più df e meno IDF

In [10]:
print(f'{"Termine":<20} {"df":>8}  {"IDF":>8}')
print('-' * 40)

campione_termini = ['film', 'american', 'nba', 'broadcast', 'playoff', 'cuckoo']
righe = []
for t_grezzo in campione_termini:
    stem = preprocessore.elabora(t_grezzo)
    if not stem:
        continue
    s = stem[0]
    df = indice.documenti_con_termine(s)
    idf = calcola_idf(s, indice)
    righe.append((s, df, idf))

for s, df, idf in sorted(righe, key=lambda x: -x[1]):
    print(f'  {s:<20} {df:>8,}  {idf:>8.4f}')

Termine                    df       IDF
----------------------------------------
  american                6,204    0.8395
  film                    5,425    0.9736
  broadcast                 529    3.3005
  nba                       138    4.6416
  playoff                   110    4.8675
  cuckoo                     15    6.8316


# TF(Term Frequency) normalizzato

BM25 non usa la term frequency grezza ma una versione satura così che all'aumentare della term frequency il contributo cresce sempre meno.

In [14]:
K1 = 1.5
B = 0.75

In [15]:
def calcola_tf_bm25(tf: int, lung_doc: int, avgdl: float, k1: float = K1, b: float = B) -> float:
    normalizzazione = 1 - b + b * (lung_doc / avgdl)
    return (tf * (k1 + 1)) / (tf + k1 * normalizzazione)

In [16]:
avgdl = indice.lunghezza_media_documenti()
lung_media = int(avgdl)

print(f'Effetto di saturazione')
print(f'  {"tf grezzo":<12}  {"tf BM25":<10}  incremento')
print('-' * 40)
tf_prec = 0
for tf in [1, 2, 3, 5, 10, 20, 50]:
    tf_bm = calcola_tf_bm25(tf, lung_media, avgdl)
    incremento = tf_bm - tf_prec
    print(f'  {tf:<12}  {tf_bm:<10.4f}  +{incremento:.4f}')
    tf_prec = tf_bm

Effetto di saturazione
  tf grezzo     tf BM25     incremento
----------------------------------------
  1             1.0005      +1.0005
  2             1.4291      +0.4286
  3             1.6671      +0.2380
  5             1.9234      +0.2563
  10            2.1741      +0.2507
  20            2.3257      +0.1516
  50            2.4272      +0.1015


In [17]:
print(f'Effetto della lunghezza del documento')
print(f'  {"lunghezza doc":<18}  {"tf BM25"}')
print('-' * 35)
for moltiplicatore in [0.25, 0.5, 1.0, 2.0, 4.0]:
    lung = int(avgdl * moltiplicatore)
    tf_bm = calcola_tf_bm25(3, lung, avgdl)
    print(f'  {lung:<18}  {tf_bm:.4f}   ({moltiplicatore}x avgdl)')

Effetto della lunghezza del documento
  lunghezza doc       tf BM25
-----------------------------------
  30                  2.0528   (0.25x avgdl)
  60                  1.9073   (0.5x avgdl)
  121                 1.6671   (1.0x avgdl)
  242                 1.3339   (2.0x avgdl)
  484                 0.9530   (4.0x avgdl)


# BM25 completo su una query

Si combina IDF e TF per calcolare il punteggio BM25 di tutti i documenti rilevanti per una query

In [19]:
def calcola_bm25_query(query: str, indice: IndiceInvertito, preprocessore: Preprocessore, k1: float = K1, b: float = B) -> dict:
    token_query = preprocessore.elabora(query)
    if not token_query:
        return {}
    avgdl = indice.lunghezza_media_documenti()
    punteggi = {}

    for termine in token_query:
        idf = calcola_idf(termine, indice)
        posting = indice.cerca(termine)
        for id_doc, tf in posting.items():
            lung_doc = indice.lunghezza_documento(id_doc)
            tf_norm = calcola_tf_bm25(tf, lung_doc, avgdl, k1, b)
            contributo = idf * tf_norm
            punteggi[id_doc] = punteggi.get(id_doc, 0.0) + contributo
    return punteggi

Test BM25 su query reali dal dataset FEVER

In [20]:
query_test = [
    'Nikolaj Coster-Waldau worked with Fox Broadcasting Company',
    'Barack Obama was born in Hawaii',
    'The NBA season was won by the Boston Celtics',
]
for query in query_test:
    punteggi = calcola_bm25_query(query, indice, preprocessore)
    top5 = sorted(punteggi.items(), key=lambda x: -x[1])[:5]
    print(f'\nQuery: "{query}"')
    print(f'  Token preprocessati: {preprocessore.elabora(query)}')
    print(f'  Documenti candidati: {len(punteggi):,}')
    print(f'  Top 5:')
    for id_doc, score in top5:
        print(f'    {id_doc:<50} score={score:.4f}')


Query: "Nikolaj Coster-Waldau worked with Fox Broadcasting Company"
  Token preprocessati: ['nikolaj', 'costerwaldau', 'work', 'fox', 'broadcast', 'compani']
  Documenti candidati: 3,637
  Top 5:
    New_Amsterdam_-LRB-TV_series-RRB-                  score=24.5750
    Nikolaj_Coster-Waldau                              score=22.9146
    A_Thousand_Times_Good_Night                        score=21.4439
    Fox_Broadcasting_Company                           score=17.8471
    Nightwatch_-LRB-1994_film-RRB-                     score=17.1365

Query: "Barack Obama was born in Hawaii"
  Token preprocessati: ['barack', 'obama', 'born', 'hawaii']
  Documenti candidati: 2,848
  Top 5:
    Presidency_of_Barack_Obama                         score=23.5455
    Barack_Obama_Presidential_Center                   score=22.1637
    Michelle_Obama                                     score=20.1088
    Barack_Obama                                       score=19.8703
    Hillary_Clinton_presidential_campaign

# Metadati (Authority e Freshness Score)

Il ranking esteso aggiunge due metadati al punteggio BM25:
* authority_score
* freshness_score

Entrambi vanno normalizzati nell'intervallo [0,1] sull'intera collezione per renderli confrontabili con il punteggio BM25

In [22]:
ALPHA = 0.3   # peso authority_score
BETA = 0.2   # peso freshness_score

In [23]:
def calcola_punteggi_metadati(corpus_arricchito: dict) -> tuple[dict, dict]:
    titoli = list(corpus_arricchito.keys())
    link_grezzi = np.array([corpus_arricchito[t]['numero_link_entrata'] for t in titoli], dtype=float)
    DATA_RIFERIMENTO = datetime(2004, 1, 1)
    def ts_in_giorni(ts_str):
        try:
            dt = datetime.strptime(ts_str, '%Y-%m-%dT%H:%M:%SZ')
            return max(0, (dt - DATA_RIFERIMENTO).days)
        except Exception:
            return 0

    giorni_grezzi = np.array([ts_in_giorni(corpus_arricchito[t]['timestamp']) for t in titoli], dtype=float)

    def normalizza(valori: np.ndarray) -> np.ndarray:
        minimo  = valori.min()
        massimo = valori.max()
        if massimo == minimo:
            return np.full_like(valori, 0.5)
        return (valori - minimo) / (massimo - minimo)

    authority_norm = normalizza(link_grezzi)
    freshness_norm = normalizza(giorni_grezzi)

    authority_scores = dict(zip(titoli, authority_norm.tolist()))
    freshness_scores = dict(zip(titoli, freshness_norm.tolist()))

    return authority_scores, freshness_scores

In [24]:
print('Calcolo metadati')
authority_scores, freshness_scores = calcola_punteggi_metadati(corpus_arricchito)
print(f'Authority scores calcolati: {len(authority_scores):,}')
print(f'Freshness scores calcolati: {len(freshness_scores):,}')

print()
print('Top 10 authority score:')
top_auth = sorted(authority_scores.items(), key=lambda x: -x[1])[:10]
for doc, score in top_auth:
    link = corpus_arricchito[doc]['numero_link_entrata']
    print(f'  {doc:<45} score={score:.4f}  link={link}')

Calcolo metadati
Authority scores calcolati: 14,364
Freshness scores calcolati: 14,364

Top 10 authority score:
  India                                         score=1.0000  link=872
  California                                    score=0.9484  link=827
  Canada                                        score=0.7867  link=686
  London                                        score=0.7706  link=672
  France                                        score=0.7500  link=654
  NBC                                           score=0.7317  link=638
  HBO                                           score=0.6491  link=566
  Europe                                        score=0.6227  link=543
  WWE                                           score=0.6021  link=525
  England                                       score=0.5814  link=507


# Ranking BM25 esteso

Combina BM25 con authority e freshness

In [25]:
def recupera_top_k(query: str, indice: IndiceInvertito, preprocessore: Preprocessore, authority_scores: dict, freshness_scores: dict, k: int   = 10, alpha: float = ALPHA, beta: float = BETA, k1: float = K1, b: float = B) -> list[tuple[str, float]]:

    punteggi_bm25 = calcola_bm25_query(query, indice, preprocessore, k1, b)
    if not punteggi_bm25:
        return []
    punteggi_finali = {}
    for id_doc, score_bm25 in punteggi_bm25.items():
        auth = authority_scores.get(id_doc, 0.0)
        fresh = freshness_scores.get(id_doc, 0.0)
        punteggi_finali[id_doc] = score_bm25 + alpha * auth + beta * fresh
    return sorted(punteggi_finali.items(), key=lambda x: -x[1])[:k]

Test BM25 esteso

In [26]:
print('Test BM25 esteso')

for query in query_test:
    risultati = recupera_top_k(query, indice, preprocessore, authority_scores, freshness_scores, k=10)
    print(f'\nQuery: "{query}"')
    for rank, (id_doc, score) in enumerate(risultati, 1):
        auth  = authority_scores.get(id_doc, 0.0)
        fresh = freshness_scores.get(id_doc, 0.0)
        print(f'  {rank:2}. {id_doc:<50} score={score:.4f} '
              f'(auth={auth:.2f}, fresh={fresh:.2f})')

Test BM25 esteso

Query: "Nikolaj Coster-Waldau worked with Fox Broadcasting Company"
   1. New_Amsterdam_-LRB-TV_series-RRB-                  score=24.7638 (auth=0.00, fresh=0.94)
   2. Nikolaj_Coster-Waldau                              score=23.1139 (auth=0.00, fresh=1.00)
   3. A_Thousand_Times_Good_Night                        score=21.6347 (auth=0.00, fresh=0.95)
   4. Fox_Broadcasting_Company                           score=18.0447 (auth=0.00, fresh=0.99)
   5. Nightwatch_-LRB-1994_film-RRB-                     score=17.3303 (auth=0.00, fresh=0.97)
   6. Headhunters_-LRB-film-RRB-                         score=16.6528 (auth=0.00, fresh=0.96)
   7. Game_of_Thrones_-LRB-season_5-RRB-                 score=14.9348 (auth=0.00, fresh=1.00)
   8. Jaime_Lannister                                    score=14.8032 (auth=0.00, fresh=1.00)
   9. Davis_Entertainment                                score=14.3843 (auth=0.00, fresh=0.97)
  10. Game_of_Thrones_-LRB-season_7-RRB-                 sc

# Confronto BM25 base vs BM25 esteso

In [27]:
query_confronto = 'The NBA season was won by the Boston Celtics'

punteggi_base = calcola_bm25_query(query_confronto, indice, preprocessore)
top_base = sorted(punteggi_base.items(), key=lambda x: -x[1])[:10]

top_esteso = recupera_top_k(query_confronto, indice, preprocessore, authority_scores, freshness_scores, k=10)

print(f'CONFRONTO RANKING — Query: "{query_confronto}"')
print()
print(f'  {"RANK":<5} {"BM25 BASE":<45} {"BM25 ESTESO"}')
print('-' * 95)
for i, ((doc_base, s_base), (doc_est, s_est)) in enumerate(zip(top_base, top_esteso), 1):
    cambio = '↑' if doc_est != doc_base else ' '
    print(f'  {i:<5} {doc_base:<45} {cambio} {doc_est}')

CONFRONTO RANKING — Query: "The NBA season was won by the Boston Celtics"

  RANK  BM25 BASE                                     BM25 ESTESO
-----------------------------------------------------------------------------------------------
  1     1984–85_NBA_season                              1984–85_NBA_season
  2     1983–84_NBA_season                              1983–84_NBA_season
  3     1985–86_NBA_season                              1985–86_NBA_season
  4     Boston_Celtics                                  Boston_Celtics
  5     1981_NBA_Finals                                 1981_NBA_Finals
  6     Rajon_Rondo                                     Rajon_Rondo
  7     Paul_Pierce                                     Paul_Pierce
  8     1968_NBA_Playoffs                               1968_NBA_Playoffs
  9     2006–07_Cleveland_Cavaliers_season              2006–07_Cleveland_Cavaliers_season
  10    2010_NBA_Finals                                 2010_NBA_Finals


Test finale per le claim SUPPORTS del campione

In [29]:
def converti_lista(valore):
    try:
        return ast.literal_eval(str(valore))
    except Exception:
        return []

In [30]:
K_VALORI = [1, 3, 5, 10]
hits = {k: 0 for k in K_VALORI}
totale = 0
CLAIM_DA_TESTARE = 200

In [34]:
df_campione['url_documenti'] = df_campione['url_documenti'].apply(converti_lista)

for _, riga in df_campione[df_campione['label'] == 'SUPPORTS'].head(CLAIM_DA_TESTARE).iterrows():
    documenti_attesi = riga['url_documenti']
    if not documenti_attesi:
        continue
    risultati = recupera_top_k(riga['testo_claim'], indice, preprocessore, authority_scores, freshness_scores, k=10)
    id_recuperati = [doc for doc, _ in risultati]

    for k in K_VALORI:
        top_k_docs = set(id_recuperati[:k])
        if any(doc in top_k_docs for doc in documenti_attesi):
            hits[k] += 1
    totale += 1

print(f'RECALL — documento ground truth trovato nei top-K (su {totale} claim SUPPORTS)')
for k in K_VALORI:
    recall = hits[k] / totale * 100 if totale > 0 else 0
    barra  = '█' * int(recall / 3)
    print(f'  Recall {k:<3}: {recall:5.1f}%  {barra}')

RECALL — documento ground truth trovato nei top-K (su 800 claim SUPPORTS)
  Recall 1  :  79.5%  ██████████████████████████
  Recall 3  :  95.5%  ███████████████████████████████
  Recall 5  :  98.0%  ████████████████████████████████
  Recall 10 :  99.0%  █████████████████████████████████
